In [ ]:
!git clone https://github.com/Jun1801/CoDraft-Bench.git

In [ ]:
%cd "YOUR_WORKING_DIR"
CD_PATH = "YOUR WORKING DIR"

In [ ]:
%pip install -r requirements.txt

In [ ]:
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if CD_PATH not in sys.path:
    sys.path.append(CD_PATH)

In [ ]:
from copy import deepcopy
import pandas as pd

from config import *
from preprocess import *
from model import *
from model.models import *
from utils import *


In [ ]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
INPUT_ROOT = "dataset"
WORK_DIR = "working"
MODEL_NAME = "dunzhang/stella_en_400M_v5"
MODEL_TYPE = "siamese"
CHECKPOINT_FILE = f"{WORK_DIR}/{MODEL_NAME}_best_model.pth"
PRETRAIN_FILE = None

In [ ]:
PATH_SIMCSE = f"{WORK_DIR}/simcse_model"
PATH_FINAL = f"{WORK_DIR}/final_similarity_model"

In [ ]:
tokenizer = get_tokenizer(MODEL_NAME)

In [ ]:
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    tokenizer=tokenizer,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED,
    rebalance=True,
    train_file="train.csv",
    val_file="val.csv",
    test_file="test.csv")

In [ ]:
data_manager._all_texts()
all_texts = data_manager.get_all_texts()
df_train, df_test, df_val = data_manager.get_data()
train_dataloader, evaluator = data_manager.get_dataloaders(model_type="cross_encoder")
train_loader, val_loader = data_manager.get_dataloaders(model_type="siamese")
train_ds, val_ds, test_ds = data_manager.get_dataset()
print(df_train['label_score'].value_counts().sort_index())

In [ ]:
train_simcse(MODEL_NAME, all_texts, PATH_SIMCSE, DEVICE)

In [ ]:
model = SiameseClassifier(PATH_SIMCSE, num_classes=5)
model.to(DEVICE)
new_tokenizer = model.encoder.tokenizer
data_manager.update_siamese_tokenizer(new_tokenizer)

In [ ]:
df_train, df_val, df_test = data_manager.get_data()
train_loader, val_loader = data_manager.get_dataloaders(model_type="siamese")

In [ ]:
class_weights = compute_class_weight(df_train['label_score'])

In [ ]:
train_siamese(PATH_SIMCSE, train_loader, val_loader, class_weights, PATH_FINAL, DEVICE)

In [ ]:
test_preds, test_true = get_preds_siamese(df_test, DEVICE)
result_df = pd.DataFrame({
    "label": df_test["label_score"],
    "pred": test_preds,
})

In [ ]:
get_stats(result_df)

In [ ]:
result_df.to_csv("result_siamese.csv", index=False)

In [ ]:
zip_model_folder(PATH_SIMCSE)

In [ ]:
zip_model_folder(PATH_FINAL)